In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Performance Management : une Qiskit Function par Q-CTRL Fire Opal
*Consulte la [référence API](https://docs.quantum.ibm.com/api/functions/q-ctrl-performance-management)*

> **Note:** Les Qiskit Functions sont une fonctionnalité expérimentale réservée aux utilisateurs des plans IBM Quantum&reg; Premium, Flex et On-Prem (via l'API IBM Quantum Platform). Elles sont en version préliminaire et susceptibles d'évoluer.


<Accordion>
<AccordionItem title="Package versions">

The code on this page was developed using the following requirements.
We recommend using these versions or newer.

```
qiskit[all]~=2.3.1
qiskit-ibm-runtime~=0.45.1
```
</AccordionItem>
</Accordion>

## Vue d'ensemble
Fire Opal Performance Management permet à quiconque d'obtenir des résultats significatifs à partir d'ordinateurs quantiques à grande échelle, sans avoir besoin d'être expert en matériel quantique. Lorsque tu exécutes des circuits avec Fire Opal Performance Management, des techniques de suppression des erreurs basées sur l'IA sont automatiquement appliquées, permettant de traiter des problèmes plus vastes avec davantage de portes et de qubits. Cette approche réduit le nombre de shots nécessaires pour obtenir la bonne réponse, sans surcharge supplémentaire — ce qui se traduit par des économies significatives en termes de temps de calcul et de coût.

Performance Management supprime les erreurs et augmente la probabilité d'obtenir la bonne réponse sur du matériel bruité. En d'autres termes, il améliore le rapport signal/bruit. L'image ci-dessous montre comment la précision accrue offerte par Performance Management peut réduire le besoin de shots supplémentaires dans le cas d'un algorithme de Transformée de Fourier Quantique à 10 qubits. Avec seulement 30 shots, Q-CTRL atteint le seuil de confiance à 99 %, alors que la configuration par défaut (`QiskitRuntime` Sampler, `optimization_level`=3 et `resilience_level`=1, `ibm_sherbrooke`) en nécessite 170 000. En obtenant la bonne réponse plus vite, tu économises un temps de calcul considérable.

![Visualisation de l'amélioration du temps d'exécution](../docs/images/guides/qctrl-performance-management/achieve_more.svg)

La fonction Performance Management peut être utilisée avec n'importe quel algorithme, et tu peux facilement l'utiliser à la place des [primitives Qiskit Runtime](/guides/primitives) standard. En arrière-plan, plusieurs techniques de suppression des erreurs fonctionnent ensemble pour prévenir les erreurs au moment de l'exécution. Toutes les méthodes du pipeline Fire Opal sont préconfigurées et indépendantes des algorithmes, ce qui signifie que tu obtiens toujours les meilleures performances prêtes à l'emploi.

Pour accéder à Performance Management, [contacte Q-CTRL](https://form.typeform.com/to/uOAVDnGg?typeform-source=q-ctrl.com).
## Description
Fire Opal Performance Management propose deux modes d'exécution similaires aux primitives Qiskit Runtime, ce qui te permet de remplacer facilement les primitives standard par le Sampler et l'Estimator Q-CTRL. Le flux de travail général pour utiliser la fonction Performance Management est :
1. Définis ton circuit (et les opérateurs dans le cas de l'Estimator).
2. Exécute le circuit.
3. Récupère les résultats.

Pour réduire le bruit matériel, Fire Opal emploie un ensemble de techniques de suppression des erreurs basées sur l'IA, illustrées dans l'image suivante. Avec Fire Opal, l'ensemble du pipeline est entièrement automatisé, sans aucune configuration nécessaire.

Le pipeline Fire Opal élimine le besoin de surcharges supplémentaires, telles qu'un temps d'exécution quantique accru ou des qubits physiques supplémentaires. Note que le temps de traitement classique reste un facteur (consulte la section [Benchmarks](#benchmarks) pour des estimations ; le « Temps total » reflète à la fois le traitement classique et quantique). Contrairement à la mitigation des erreurs, qui nécessite une surcharge sous forme d'échantillonnage, la suppression des erreurs de Fire Opal opère aux niveaux des portes et des impulsions pour traiter diverses sources de bruit et prévenir la survenue d'erreurs. En prévenant les erreurs, la nécessité d'un post-traitement coûteux est éliminée.

L'image suivante illustre les méthodes de suppression des erreurs automatisées par Fire Opal Performance Management.

![Visualisation du pipeline de suppression des erreurs](../docs/images/guides/qctrl-performance-management/error_suppression.svg)

La fonction offre deux primitives, Sampler et Estimator, et les entrées et sorties des deux étendent la spécification implémentée pour les [primitives Qiskit Runtime V2](/guides/primitive-input-output#pubs).
## Benchmarks
Les résultats de [benchmarking algorithmique publiés](https://journals.aps.org/prapplied/abstract/10.1103/PhysRevApplied.20.024034) démontrent une amélioration significative des performances pour divers algorithmes, notamment Bernstein-Vazirani, la transformée de Fourier quantique, la recherche de Grover, l'algorithme d'optimisation approximative quantique (QAOA) et le solveur propre variationnel quantique (VQE). La suite de cette section fournit plus de détails sur les types d'algorithmes que tu peux exécuter, ainsi que les performances et temps d'exécution attendus.

Les études indépendantes suivantes montrent comment Performance Management de Q-CTRL permet la recherche algorithmique à une échelle record :
- [Parametrized Energy-Efficient Quantum Kernels for Network Service Fault Diagnosis](https://arxiv.org/abs/2405.09724v1) — apprentissage de noyaux quantiques jusqu'à 50 qubits
- [Tensor-based quantum phase difference estimation for large-scale demonstration](https://arxiv.org/abs/2408.04946) — estimation de phase quantique jusqu'à 33 qubits
- [Hierarchical Learning for Quantum ML: Novel Training Technique for Large-Scale Variational Quantum Circuits](https://arxiv.org/abs/2311.12929) — chargement de données quantiques jusqu'à 21 qubits

Le tableau suivant fournit un guide approximatif sur la précision et les temps d'exécution issus de benchmarkings antérieurs sur `ibm_fez`. Les performances sur d'autres appareils peuvent varier. Le temps d'utilisation est basé sur une hypothèse de 10 000 shots par circuit. Le « Nombre de qubits » indiqué n'est pas une limite stricte, mais représente des seuils approximatifs où tu peux t'attendre à une précision de solution extrêmement constante. Des problèmes de plus grande taille ont été résolus avec succès, et les tests au-delà de ces limites sont encouragés.

| Exemple    | Nombre de qubits | Précision | Mesure de précision | Temps total (s) | Utilisation Runtime (s) | Primitive (Mode) |
| ---------  | ---------------- | -------------------------- | -------- | ---------- | ------------- |------------- |
| Bernstein–Vazirani  |  50Q    | 100 %  | Taux de succès (pourcentage d'exécutions où la bonne réponse est la chaîne de bits avec le plus grand nombre de résultats)     | 10    | 8         | Sampler |
| Transformée de Fourier Quantique | 30Q              | 100 % | Taux de succès (pourcentage d'exécutions où la bonne réponse est la chaîne de bits avec le plus grand nombre de résultats)      | 10    | 8        | Sampler |
| Estimation de Phase Quantique  | 30Q   | 99,9998 %  | Précision de l'angle trouvé : `1- abs(real_angle - angle_found)/pi`  | 10  | 8  | Sampler |
| Simulation quantique : modèle d'Ising (15 étapes) | 20Q  | 99,775 %   |  $A$ (défini ci-dessous)  |  60 (par étape)  | 15 (par étape)   | Estimator |
| Simulation quantique 2 : dynamique moléculaire (20 points temporels) | 34Q  |  96,78 %  |  $A_{mean}$ (défini ci-dessous)   | 10 (par point temporel)    | 6 (par point temporel)  | Estimator |

Définition de la précision de la mesure d'une valeur d'espérance — la métrique $A$ est définie comme suit :
$$
A = 1 - \frac{|\epsilon^{ideal} - \epsilon^{meas}|}{\epsilon^{ideal}_{max} - \epsilon^{ideal}_{min}},
$$
où $ \epsilon^{ideal} $ = valeur d'espérance idéale, $ \epsilon^{meas} $ = valeur d'espérance mesurée, $\epsilon^{ideal}_{max} $ = valeur idéale maximale, et $\epsilon^{ideal}_{min}$ = valeur idéale minimale. $A_{mean}$ est simplement la moyenne de la valeur de $A$ sur plusieurs mesures.

Cette métrique est utilisée car elle est invariante aux décalages globaux et à la mise à l'échelle de la plage de valeurs atteignables. En d'autres termes, que tu décales la plage des valeurs d'espérance possibles vers le haut ou vers le bas, ou que tu augmentes l'étendue, la valeur de $A$ doit rester cohérente.
## Premiers pas
Fire Opal Performance Management utilise Qiskit v`2.0.0`, qui est la version recommandée. Les versions supportées sont Qiskit >=v`2.0.0`.
Authentifie-toi avec ta [clé API IBM Quantum Platform](http://quantum.cloud.ibm.com/), et sélectionne la Qiskit Function comme suit. (Cet extrait suppose que tu as déjà [sauvegardé ton compte](/guides/functions#install-qiskit-functions-catalog-client) dans ton environnement local.)

In [1]:
# This cell is hidden from users. It hides the "...You have imported samplomatic..." warning.
import warnings

warnings.filterwarnings("ignore", message=".*You have imported samplomatic*")

In [2]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# verify that you have access to the function
catalog.list()

[QiskitFunction(qunova/hivqe-chemistry),
 QiskitFunction(global-data-quantum/quantum-portfolio-optimizer),
 QiskitFunction(algorithmiq/tem),
 QiskitFunction(qedma/qesem),
 QiskitFunction(multiverse/singularity),
 QiskitFunction(ibm/circuit-function),
 QiskitFunction(q-ctrl/optimization-solver),
 QiskitFunction(colibritd/quick-pde),
 QiskitFunction(q-ctrl/performance-management),
 QiskitFunction(kipu-quantum/iskay-quantum-optimizer)]

In [3]:
# Access Function
perf_mgmt = catalog.load("q-ctrl/performance-management")

<Admonition type="note" title="Does this function support all IBM backends?">
If you want to use a backend that this function does not currently support, [reach out to Q-CTRL](https://form.typeform.com/to/iuujEAEI?typeform-source=q-ctrl.com) to add support.
</Admonition>

## Estimator primitive

### Estimator example

Use Fire Opal Performance Management's Estimator primitive to determine the expectation value of a single circuit-observable pair.

In addition to the `qiskit-ibm-catalog` and `qiskit` packages, you will also use the `numpy` package to run this example. You can install this package by uncommenting the following cell if you are running this example in a notebook using the IPython kernel.

In [4]:
# %pip install numpy

**2. Exécuter le circuit**

Exécute le circuit et définis éventuellement le backend et le nombre de shots.

In [5]:
import numpy as np
from qiskit.circuit.library import iqp
from qiskit.quantum_info import random_hermitian, SparsePauliOp

n_qubits = 50

# Generate a random circuit
mat = np.real(random_hermitian(n_qubits, seed=1234))
circuit = iqp(mat)
circuit.measure_all()

# Define observables as a string
observable = SparsePauliOp("Z" * n_qubits)

In [6]:
# Create PUB tuple
estimator_pubs = [(circuit, observable)]

Tu peux utiliser les [APIs Qiskit Serverless](/guides/serverless) pour vérifier le statut de ta charge de travail Qiskit Function :

In [7]:
# This cell is hidden from users
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend_name = service.least_busy().name

In [8]:
# Run the circuit using Estimator
qctrl_estimator_job = perf_mgmt.run(
    primitive="estimator",
    pubs=estimator_pubs,
    backend_name=backend_name,
)

**3. Récupérer le résultat**

In [9]:
qctrl_estimator_job.status()

'QUEUED'

Les résultats ont le même format qu'un [résultat Estimator](/guides/estimator-input-output) :

In [10]:
# Retrieve the counts from the result list
result = qctrl_estimator_job.result()

The results have the same format as an [Estimator result](/docs/guides/estimator-input-output):

In [22]:
import numpy

result_str = str(result)

with numpy.printoptions(threshold=200):
    print(
        f"The result of the submitted job had {len(result)} PUB "
        f"and has a value:\n {result[0]}\n"
    )

print("The associated PubResult of this job has the following DataBins:")
print(f"{result[0].data}\n")

print(f"And this DataBin has attributes: {result[0].data.keys()}")

print("The expectation values measured from this PUB are:")
print(f"{result[0].data.evs}")

The result of the submitted job had 1 PUB
The result of the submitted job had 1 PUB and has a value:
 PubResult(data=DataBin(evs=0.0195, stds=0.9998098569228051), metadata={'precision': None})

The associated PubResult of this job has the following DataBins:
DataBin(evs=0.0195, stds=0.9998098569228051)

And this DataBin has attributes: dict_keys(['evs', 'stds'])
The expectation values measured from this PUB are:
0.0195


## Primitive Sampler
### Exemple avec le Sampler
Utilise la primitive Sampler de Fire Opal Performance Management pour exécuter un circuit Bernstein–Vazirani. Cet algorithme, utilisé pour trouver une chaîne cachée à partir des sorties d'une fonction boîte noire, est un algorithme de benchmarking courant car il n'existe qu'une seule réponse correcte.
**1. Créer le circuit**

Définis la bonne réponse à l'algorithme, la chaîne de bits cachée, et le circuit Bernstein–Vazirani. Tu peux ajuster la largeur du circuit en modifiant simplement `circuit_width`.

In [12]:
import qiskit

circuit_width = 35
hidden_bitstring = "1" * circuit_width

# Create circuit, reserving one qubit for BV oracle
bv_circuit = qiskit.QuantumCircuit(circuit_width + 1, circuit_width)
bv_circuit.x(circuit_width)
bv_circuit.h(range(circuit_width + 1))
for input_qubit, bit in enumerate(reversed(hidden_bitstring)):
    if bit == "1":
        bv_circuit.cx(input_qubit, circuit_width)
bv_circuit.barrier()
bv_circuit.h(range(circuit_width + 1))
bv_circuit.barrier()
for input_qubit in range(circuit_width):
    bv_circuit.measure(input_qubit, input_qubit)

# Create PUB tuple
sampler_pubs = [(bv_circuit,)]

**2. Exécuter le circuit**

Exécute le circuit et définis éventuellement le backend et le nombre de shots.

In [13]:
# Run the circuit using Sampler
qctrl_sampler_job = perf_mgmt.run(
    primitive="sampler",
    pubs=sampler_pubs,
    backend_name=backend_name,
)

Vérifie le [statut](/guides/functions#check-job-status) de ta charge de travail Qiskit Function ou récupère les [résultats](/guides/functions#retrieve-results) comme suit :

In [14]:
# Print the ID so you can use it later, if necessary
print(qctrl_sampler_job.job_id)

qctrl_sampler_job.status()

60fe2fa1-a860-43e4-8615-c6ac4180f93b


'QUEUED'

**3. Retrieve the result**

In [15]:
# Retrieve the job results
sampler_result = qctrl_sampler_job.result()

In [16]:
# Get results for the first (and only) PUB
pub_result = sampler_result[0]
counts = pub_result.data.c.get_counts()

print("Counts for the meas output register (limited to 30 results):")
for i, (bitstring, count) in enumerate(counts.items()):
    if i >= 50:
        print(f"  ... ({len(counts) - 30} more items)")
        break
    print(f"  {bitstring}: {count}")

Counts for the meas output register (limited to 30 results):
  11111111111111111111111111111111111: 1661
  11111111111111111111111111110111111: 60
  11111111111111111111111111111101111: 54
  11111111111111111111111111111110111: 54
  11111111111111011111111111111111111: 46
  11111111111111111110111111111111111: 44
  11111111111111111111111101111111111: 42
  11111111111111111111111110111111111: 42
  11111111111111110111111111111111111: 41
  11111111111111111111111111111111101: 39
  11111111111111111111101111111111111: 38
  11111111111111111111110111111111111: 38
  11111111111111111111111111101111111: 37
  11111111111111111111111111111111110: 36
  11111111111110111111111111111111111: 35
  11111111111111111111111111111011111: 32
  11111111111111101111111111111111111: 32
  01111111111111111111111111111111111: 27
  11111111111111111011111111111111111: 23
  11111111101111111111111111111111111: 22
  11111111111111111111111111111111011: 21
  11111111011111111111111111111111111: 20
  00000000000

**3. Plot the top bitstrings**

Plot the bitstring with the highest counts to see if the hidden bitstring was the mode.

In [17]:
import matplotlib.pyplot as plt


def plot_top_bitstrings(counts_dict, hidden_bitstring=None):
    # Sort and take the top 100 bitstrings
    top_100 = sorted(counts_dict.items(), key=lambda x: x[1], reverse=True)[
        :100
    ]
    if not top_100:
        print("No bitstrings found in the input dictionary.")
        return

    # Unzip the bitstrings and their counts
    bitstrings, counts = zip(*top_100)

    # Assign colors: purple if the bitstring matches hidden_bitstring,
    # otherwise gray
    colors = [
        "#680CE9" if bit == hidden_bitstring else "gray" for bit in bitstrings
    ]

    # Create the bar plot
    plt.figure(figsize=(15, 8))
    plt.bar(
        range(len(bitstrings)), counts, tick_label=bitstrings, color=colors
    )

    # Rotate the bitstrings for better readability
    plt.xticks(rotation=90, fontsize=8)
    plt.xlabel("Bitstrings")
    plt.ylabel("Counts")
    plt.title("Top 100 Bitstrings by Counts")

    # Show the plot
    plt.tight_layout()
    plt.show()

The hidden bitstring is highlighted in purple, and it should be the bitstring with the highest number of counts.

In [18]:
plot_top_bitstrings(counts, hidden_bitstring)

<Image src="../docs/images/guides/q-ctrl-performance-management/extracted-outputs/8106d906-0.avif" alt="Output of the previous code cell" />

**3. Tracer les principales chaînes de bits**

Trace la chaîne de bits avec le plus grand nombre de résultats pour vérifier si la chaîne cachée est bien le mode.